### 8-Puzzle A* Search Implementation
This implementation uses:
- **$g(n)$**: The path cost from the start node to node $n$ (represented by the depth of the node in the search tree).
- **$h(n)$**: The Manhattan distance heuristic (the sum of the horizontal and vertical distances of the tiles from their goal positions).

In [ ]:
import heapq

class PuzzleState:
    def __init__(self, board, parent=None, move="", depth=0):
        self.board = board
        self.parent = parent
        self.move = move
        self.depth = depth
        self.h = self.calculate_manhattan()
        self.f = self.depth + self.h

    def calculate_manhattan(self):

        goal_positions = {
            1: (0, 0), 2: (0, 1), 3: (0, 2),
            4: (1, 0), 5: (1, 1), 6: (1, 2),
            7: (2, 0), 8: (2, 1), 0: (2, 2)
        }
        distance = 0
        for i, val in enumerate(self.board):
            if val != 0:
                curr_row, curr_col = i // 3, i % 3
                goal_row, goal_col = goal_positions[val]
                distance += abs(curr_row - goal_row) + abs(curr_col - goal_col)
        return distance

    def get_blank_index(self):
        return self.board.index(0)

    def get_neighbors(self):
        neighbors = []
        blank = self.get_blank_index()
        row, col = blank // 3, blank % 3

        moves = {
            'UP': (-1, 0),
            'DOWN': (1, 0),
            'LEFT': (0, -1),
            'RIGHT': (0, 1)
        }

        for dir_name, (dr, dc) in moves.items():
            new_row, new_col = row + dr, col + dc
            if 0 <= new_row < 3 and 0 <= new_col < 3:
                new_blank = new_row * 3 + new_col
                new_board = list(self.board)
                new_board[blank], new_board[new_blank] = new_board[new_blank], new_board[blank]
                neighbors.append(PuzzleState(tuple(new_board), self, dir_name, self.depth + 1))
        return neighbors

    def __lt__(self, other):
        return self.f < other.f

    def __eq__(self, other):
        return self.board == other.board

    def __hash__(self):
        return hash(self.board)

In [ ]:
def solve_a_star(start_board):
    goal_board = (1, 2, 3, 4, 5, 6, 7, 8, 0)
    start_state = PuzzleState(start_board)

    open_set = []
    heapq.heappush(open_set, start_state)
    closed_set = set()

    nodes_expanded = 0

    while open_set:
        current_state = heapq.heappop(open_set)
        nodes_expanded += 1

        if current_state.board == goal_board:
            path = []
            curr = current_state
            while curr:
                path.append(curr)
                curr = curr.parent
            return path[::-1], nodes_expanded

        closed_set.add(current_state.board)

        for neighbor in current_state.get_neighbors():
            if neighbor.board in closed_set:
                continue
            heapq.heappush(open_set, neighbor)

    return None, nodes_expanded

In [ ]:
def print_board(board):
    for i in range(0, 9, 3):
        print(list(board[i:i+3]))

# Initial state: 0 represents the blank tile
# This initial board is standard but solvable
initial_board = (1, 2, 3, 0, 4, 6, 7, 5, 8)

print("Initial State:")
print_board(initial_board)
print("-" * 20)

path, expanded = solve_a_star(initial_board)

if path:
    for step, state in enumerate(path):
        print(f"Step {step}: Action = '{state.move if state.move else 'START'}'")
        print(f"g(n) = {state.depth}, h(n) = {state.h}, f(n) = {state.f}")
        print_board(state.board)
        print("-" * 20)
    print(f"Solution found in {len(path) - 1} moves. Nodes expanded: {expanded}")
else:
    print("No solution found.")

Initial State:
[1, 2, 3]
[0, 4, 6]
[7, 5, 8]
--------------------
Step 0: Action = 'START'
g(n) = 0, h(n) = 3, f(n) = 3
[1, 2, 3]
[0, 4, 6]
[7, 5, 8]
--------------------
Step 1: Action = 'RIGHT'
g(n) = 1, h(n) = 2, f(n) = 3
[1, 2, 3]
[4, 0, 6]
[7, 5, 8]
--------------------
Step 2: Action = 'DOWN'
g(n) = 2, h(n) = 1, f(n) = 3
[1, 2, 3]
[4, 5, 6]
[7, 0, 8]
--------------------
Step 3: Action = 'RIGHT'
g(n) = 3, h(n) = 0, f(n) = 3
[1, 2, 3]
[4, 5, 6]
[7, 8, 0]
--------------------
Solution found in 3 moves. Nodes expanded: 4
